 # Scenario: Fine‑tune ResNet‑50 for Music Genre Classification
Context:
A streaming service wants to automatically classify songs into genres (e.g., rock, jazz, classical, hip‑hop, electronic). They have 4,000 audio tracks labeled by genre. Instead of training from scratch, they’ll fine‑tune a ResNet‑50 pretrained on ImageNet, but adapted to work with spectrogram images of audio.

📊 Dataset
- Convert each audio track into a Mel‑spectrogram (visual representation of sound frequencies over time).
- Each spectrogram is treated like an image (RGB channels).
- Dataset: 4,000 spectrograms across 5 genres.

In [1]:
# Fine tuning a pretrained model
# Scenario: Fine-tune ResNet-50 pretrained on ImageNet
# to classify 5 music genres using spectrogram images.

# Import PyTorch library
import torch

# Import torchvision for pretrained models
import torchvision

# Import neural network module
import torch.nn as nn

# Import pretrained models and transformations
from torchvision import models, transforms

# Import dataset loader and DataLoader
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

import os
from PIL import Image
import numpy as np # Import numpy for creating image data
import shutil # Import shutil for removing directories


# ---------------------------------------------------------
# STEP 1 : Load Pretrained ResNet-50 Model
# ---------------------------------------------------------

# Load ResNet-50 pretrained on ImageNet
model = models.resnet50(weights='IMAGENET1K_V2')

# Transfer learning:
# The model already knows basic visual features
# such as edges, textures, and patterns.


# ---------------------------------------------------------
# STEP 2 : Freeze All Layers
# ---------------------------------------------------------

# Prevent pretrained weights from updating

for param in model.parameters():
    param.requires_grad = False


# ---------------------------------------------------------
# STEP 3 : Replace Final Classifier
# ---------------------------------------------------------

# We have 5 music genres
num_classes = 5

model.fc = nn.Sequential(

    # Dropout reduces overfitting
    nn.Dropout(0.4),

    # First fully connected layer
    nn.Linear(model.fc.in_features, 256),

    # Activation function
    nn.ReLU(),

    # Final layer for genre classification
    nn.Linear(256, num_classes)
)

# New classifier structure
#
# 2048 features
#     ↓
# Dropout
#     ↓
# Linear (2048 → 256)
#     ↓
# ReLU
#     ↓
# Linear (256 → 5 genres)


# ---------------------------------------------------------
# STEP 4 : Unfreeze Last Residual Block (Fine-Tuning)
# ---------------------------------------------------------

# Allow the deepest layer to adapt to spectrogram patterns

for param in model.layer4.parameters():
    param.requires_grad = True


# ---------------------------------------------------------
# STEP 5 : Differential Learning Rates
# ---------------------------------------------------------

optimizer = torch.optim.Adam([

    # Small learning rate for pretrained layer4
    {'params': model.layer4.parameters(), 'lr': 1e-4},

    # Higher learning rate for classifier head
    {'params': model.fc.parameters(), 'lr': 1e-3},

])


# ---------------------------------------------------------
# STEP 6 : Loss Function
# ---------------------------------------------------------

# CrossEntropyLoss is used for multi-class classification
criterion = nn.CrossEntropyLoss()


# ---------------------------------------------------------
# STEP 7 : Image Transformations
# ---------------------------------------------------------

# Spectrograms are treated as images

transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])


# ---------------------------------------------------------
# STEP 8 : Load Spectrogram Dataset
# ---------------------------------------------------------

# Example dataset structure:
#
# spectrogram_dataset/
#     rock/
#     jazz/
#     classical/
#     hiphop/
#     electronic/

# Create dummy directories and dummy image files to prevent errors
dataset_root = "spectrogram_dataset"
classes = ["rock", "jazz", "classical", "hiphop", "electronic"]

# Remove existing dummy data directory to ensure a fresh start
if os.path.exists(dataset_root):
    shutil.rmtree(dataset_root)

for cls in classes:
    class_path = os.path.join(dataset_root, cls)
    os.makedirs(class_path, exist_ok=True)
    # Create a small, valid dummy PNG image in each class folder
    dummy_image_path = os.path.join(class_path, f'dummy_{cls}.png') # Changed to PNG
    # Create a simple numpy array for image data
    dummy_array = np.random.randint(0, 256, (64, 64, 3), dtype=np.uint8)
    dummy_img = Image.fromarray(dummy_array, 'RGB') # Ensure it's an RGB image
    dummy_img.save(dummy_image_path, format='PNG') # Explicitly save as PNG

dataset = ImageFolder(dataset_root, transform=transform)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)


# ---------------------------------------------------------
# STEP 9 : Training Loop
# ---------------------------------------------------------

epochs = 10

for epoch in range(epochs):

    running_loss = 0

    for images, labels in dataloader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print("Epoch:", epoch+1, "Loss:", running_loss)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 154MB/s]
/tmp/ipykernel_831/3962179504.py:164: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  dummy_img = Image.fromarray(dummy_array, 'RGB') # Ensure it's an RGB image


Epoch: 1 Loss: 1.619952917098999
Epoch: 2 Loss: 1.424640417098999
Epoch: 3 Loss: 1.1628077030181885
Epoch: 4 Loss: 0.8432056307792664
Epoch: 5 Loss: 0.5193175673484802
Epoch: 6 Loss: 0.2819378674030304
Epoch: 7 Loss: 0.11346006393432617
Epoch: 8 Loss: 0.05201105400919914
Epoch: 9 Loss: 0.020752564072608948
Epoch: 10 Loss: 0.00876428373157978
